<a href="https://colab.research.google.com/github/dhanushkaputty/DL/blob/main/week12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

text = "hello world"
chars = sorted(list(set(text)))
char2idx = {c:i for i,c in enumerate(chars)}
idx2char = {i:c for c,i in char2idx.items()}

seq_len = 4
X, y = [], []

for i in range(len(text)-seq_len):
    X.append([char2idx[c] for c in text[i:i+seq_len]])
    y.append(char2idx[text[i+seq_len]])

X = torch.tensor(X)
y = torch.tensor(y)

class RNNModel(nn.Module):
    def __init__(self, vocab, hidden):
        super().__init__()
        self.emb = nn.Embedding(vocab, hidden)
        self.rnn = nn.RNN(hidden, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab)

    def forward(self, x):
        x = self.emb(x)
        out,_ = self.rnn(x)
        return self.fc(out[:,-1,:])

model = RNNModel(len(chars), 128)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

for i in range(300):
    out = model(X)
    loss = loss_fn(out, y)
    opt.zero_grad(); loss.backward(); opt.step()

def predict(s):
    x = torch.tensor([[char2idx[c] for c in s]])
    out = model(x)
    return idx2char[torch.argmax(out).item()]

print("RNN → hell:", predict("hell"))

RNN → hell: o


LSTM (Long-Term Dependencies)

In [2]:
class LSTMModel(nn.Module):
    def __init__(self, vocab, hidden):
        super().__init__()
        self.emb = nn.Embedding(vocab, hidden)
        self.lstm = nn.LSTM(hidden, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab)

    def forward(self, x):
        x = self.emb(x)
        out,_ = self.lstm(x)
        return self.fc(out[:,-1,:])

model = LSTMModel(len(chars), 128)

for i in range(300):
    out = model(X)
    loss = loss_fn(out, y)
    opt.zero_grad(); loss.backward(); opt.step()

print("LSTM → hell:", predict("hell"))

LSTM → hell: o


GRU

In [3]:
class GRUModel(nn.Module):
    def __init__(self, vocab, hidden):
        super().__init__()
        self.emb = nn.Embedding(vocab, hidden)
        self.gru = nn.GRU(hidden, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab)

    def forward(self, x):
        x = self.emb(x)
        out,_ = self.gru(x)
        return self.fc(out[:,-1,:])

model = GRUModel(len(chars), 128)

for i in range(300):
    out = model(X)
    loss = loss_fn(out, y)
    opt.zero_grad(); loss.backward(); opt.step()

print("GRU → hell:", predict("hell"))

GRU → hell: w


. Word Prediction

In [4]:
text = "i love deep learning i love ai"
words = text.split()

w2i = {w:i for i,w in enumerate(set(words))}
i2w = {i:w for w,i in w2i.items()}

X, y = [], []
for i in range(len(words)-3):
    X.append([w2i[w] for w in words[i:i+3]])
    y.append(w2i[words[i+3]])

X = torch.tensor(X)
y = torch.tensor(y)

class WordModel(nn.Module):
    def __init__(self, vocab, hidden):
        super().__init__()
        self.emb = nn.Embedding(vocab, hidden)
        self.lstm = nn.LSTM(hidden, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab)

    def forward(self, x):
        x = self.emb(x)
        out,_ = self.lstm(x)
        return self.fc(out[:,-1,:])

model = WordModel(len(words), 128)

for i in range(300):
    out = model(X)
    loss = loss_fn(out, y)
    opt.zero_grad(); loss.backward(); opt.step()

def predict_word(s):
    x = torch.tensor([[w2i[w] for w in s.split()]])
    out = model(x)
    return i2w[torch.argmax(out).item()]

print("Word → i love deep:", predict_word("i love deep"))

Word → i love deep: deep


Encoder–Decoder

In [5]:
pairs = [("hi", "hello"), ("bye", "goodbye")]

input_vocab = {"hi":0, "bye":1}
output_vocab = {"hello":0, "goodbye":1}
inv_out = {0:"hello", 1:"goodbye"}

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(2, 8)
        self.lstm = nn.LSTM(8, 8)

    def forward(self, x):
        x = self.emb(x)
        _, (h,c) = self.lstm(x)
        return h,c

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(2, 8)
        self.lstm = nn.LSTM(8, 8)
        self.fc = nn.Linear(8, 2)

    def forward(self, x, h, c):
        x = self.emb(x)
        out,(h,c) = self.lstm(x,(h,c))
        return self.fc(out), h, c

enc = Encoder()
dec = Decoder()

def translate(word):
    x = torch.tensor([[input_vocab[word]]])
    h,c = enc(x)
    y = torch.tensor([[0]])
    out,_,_ = dec(y,h,c)
    return inv_out[torch.argmax(out).item()]

print("Translate hi →", translate("hi"))

Translate hi → goodbye


Attention Mechanism

In [6]:
import torch.nn.functional as F

scores = torch.tensor([2.0, 1.0, 0.5])
weights = F.softmax(scores, dim=0)

print("Attention Weights:", weights)

Attention Weights: tensor([0.6285, 0.2312, 0.1402])
